<a href="https://colab.research.google.com/github/Tuchobm/Curso-IA-Google-Colab/blob/main/5_1_inferencia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio 1: Inferencia de LLM

En este ejercicio, aprenderás a cargar y utilizar un modelo de lenguaje pre-entrenado de la biblioteca 'transformers' para realizar inferencia (generación de texto).

Configuraremos un pipeline de generación de texto e interactuaremos con el modelo a través de un prompt de sistema y entradas de usuario.

Además, encontrarás partes del código contendrán el comentario de '# ACTIVIDAD' que indican dónde debes completar el código o realizar tareas específicas. Asegúrate de seguir las instrucciones y completar el código donde se indique. Finalmente, en el final de este ejercicio, deberás responder a una serie de preguntas.

# Cargar el modelo y el tokenizer

In [ ]:
from transformers import pipeline

# ACTIVIDAD: Prueba diferentes modelos de lenguaje para ver cuál se adapta mejor a tus necesidades.
model_name = "unsloth/Llama-3.2-1B-Instruct"
# model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
# model_name = "Qwen/Qwen2.5-7B-Instruct-1M"

model = pipeline(
    task="text-generation",
    model=model_name,
    torch_dtype="auto",
    device_map="auto",
)

## Conversación con el modelo

In [ ]:
# ACTIVIDAD: Prueba diferentes system prompts y hyperparámetros
system_prompt = """
Eres un assistente virtual que ayuda a los usuarios a encontrar información sobre la historia de España.
Pero solo conces información sobre la historia de España antes del 1800.
Intenta siempre responder com una persona de esa época.
"""
temperature = 0.8
top_p = 0.95
max_new_tokens = 256

messages = [{"role": "system", "content": system_prompt}]
while True:
    user_input = input("Usuario (Escribe 'exit' para salir): ")
    if user_input.lower() == "exit" or user_input.lower() == "":
        break

    messages.append({"role": "user", "content": user_input})
    response = model(
        messages,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=model.tokenizer.eos_token_id,
    )
    msg = response[0]["generated_text"][-1]

    # Encuentra el pensamiento del asistente (deepseek)
    if "</think>" in msg["content"]:
        pensamiento, msg["content"] = msg["content"].split("</think>", 1)
        pensamiento, msg["content"] = pensamiento.strip(), msg["content"].strip()
        print("Pensamiento asistente:", pensamiento)
    messages.append(msg)
    print("Asistente:", msg["content"])

## Preguntas

- Describe cómo cambiar el `system_prompt` influye en la personalidad, el tono y la información que proporciona el asistente. Busca 3 ejemplos para que el modelo actúe como un asistente diferente (p. ej., un amigo, un experto en historia, un profesor). ¿Cómo cambia la calidad de las respuestas? ¿Qué tipo de preguntas funcionan mejor con cada `system_prompt`?
- Experimenta con diferentes valores para `temperature` (p. ej., 0.2, 0.7, 1.0) y `top_p` (p. ej., 0.5, 0.95). ¿Cómo afectan estos parámetros a la creatividad frente a la coherencia del texto generado? ¿Qué sucede si reduces significativamente `max_new_tokens`?
- ¿El modelo se adhiere estrictamente a las restricciones establecidas en el `system_prompt` (p. ej., salirse de su rol)? ¿Cómo responde a preguntas que no están directamente relacionadas con su rol? ¿Qué tipo de preguntas parecen funcionar mejor?
- Hazle al modelo la misma pregunta usando diferentes formulaciones (p. ej., una pregunta simple frente a una más detallada). ¿Cómo afecta la calidad o el detalle de la entrada del usuario a la respuesta del asistente?
- Continúa una conversación durante varios turnos. ¿Parece el modelo "recordar" partes anteriores de la conversación? ¿Cómo podría la ventana de contexto limitada afectar las interacciones más largas?
- ¿Qué diferencias encuentras entre los tres modelos? ¿Cuál parece ser más efectivo para la generación de texto? ¿Por qué crees que es así?